<a href="https://colab.research.google.com/github/DrewThomasson/Universal_TTS_Finetune/blob/main/notebook/universal_tts_finetune_webui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Welcome to the **Universal TTS Fine-tune WebUI** (XTTS & Piper)!

This Jupyter Notebook allows you to easily run the fine-tuning interface on Google Colab with GPU acceleration enabled. It supports fine-tuning both **Coqui XTTS v2** and **Piper** models, speech dataset preprocessing, and inference testing.

For documentation and local installation guide, visit the repository at: https://github.com/DrewThomasson/Universal_TTS_Finetune

In [ ]:
# @title 🛠️ Install Requirements
!sudo apt-get update
!sudo apt-get -y install libegl1 libopengl0 libxcb-cursor0 espeak-ng
!pip install -r https://raw.githubusercontent.com/DrewThomasson/Universal_TTS_Finetune/main/requirements.txt

In [ ]:
# @title 🚀 Run Web Interface
%cd /content/
!git clone https://github.com/DrewThomasson/Universal_TTS_Finetune.git
%cd /content/Universal_TTS_Finetune
!python web_gui.py --share

In [ ]:
# @title 📦 Zip and Upload Fine-tuned Models
import shutil
import requests
import os
from tqdm import tqdm

# Define the paths
finetune_dir = '/content/Universal_TTS_Finetune/finetune_models/ready'  # @param {type:"string"}
dataset_dir = '/content/Universal_TTS_Finetune/finetune_models/dataset'  # @param {type:"string"}

# Create a temporary directory to collect both folders before zipping
temp_dir = "/content/temp_finetune_dataset"
os.makedirs(temp_dir, exist_ok=True)

# Copy both directories into the temporary directory with a progress bar
def copy_with_progress(src, dst):
    if not os.path.exists(src):
        print(f"Warning: Path {src} does not exist. Skipping.")
        return
    total_files = sum(len(files) for _, _, files in os.walk(src))
    with tqdm(total=total_files, desc=f"Copying {os.path.basename(src)}") as pbar:
        for root, _, files in os.walk(src):
            rel_path = os.path.relpath(root, src)
            target_path = os.path.join(dst, rel_path)
            os.makedirs(target_path, exist_ok=True)
            for file in files:
                shutil.copy(os.path.join(root, file), target_path)
                pbar.update(1)

copy_with_progress(finetune_dir, os.path.join(temp_dir, "ready"))
copy_with_progress(dataset_dir, os.path.join(temp_dir, "dataset"))

# Create a zip file of the combined directories with progress
zip_filename = "finetune_and_dataset.zip"
with tqdm(total=100, desc="Zipping files") as pbar:
    shutil.make_archive("finetune_and_dataset", 'zip', root_dir=temp_dir)
    pbar.update(100)

# Define a function to stream the upload with a progress bar
def upload_with_progress(file_path, url):
    file_size = os.path.getsize(file_path)
    with open(file_path, 'rb') as f, tqdm(
        total=file_size, unit='B', unit_scale=True, desc="Uploading"
    ) as progress:
        response = requests.post(
            url,
            files={"file": (file_path, f)},
            stream=True,
            headers={"Connection": "keep-alive"},
        )
        # Update the progress bar as chunks are sent
        for chunk in response.iter_content(chunk_size=4096):
            if chunk:
                progress.update(len(chunk))
    return response

# Upload the zip file to file.io with a progress bar
response = upload_with_progress(zip_filename, "https://file.io/?expires=1d")

# Parse the response and display the download link
if response.status_code == 200:
    download_link = response.json().get('link', 'Error: No link found.')
    print(f"Your file is ready: {download_link}")
else:
    print(f"Failed to upload: {response.status_code} - {response.text}")
